# Data Cleaner: Historical Misclassification Mining

This notebook aggregates historical misclassification artifacts from `./logs` and `./wandb`, builds a hard-sample ranking, and exports actionable cleaning recommendations.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path('.').resolve()
LOG_DIR = ROOT / 'logs'
WANDB_DIR = ROOT / 'wandb'
OUT_CSV = LOG_DIR / 'datacleaner_hard_samples.csv'
OUT_RUN_METRICS = LOG_DIR / 'datacleaner_run_metrics.csv'

TOP_K = 200
DISPLAY_K = 24

print('ROOT =', ROOT)
print('LOG_DIR exists =', LOG_DIR.exists())
print('WANDB_DIR exists =', WANDB_DIR.exists())

In [ ]:
def discover_error_csvs(log_dir: Path, wandb_dir: Path):
    files = []
    if log_dir.exists():
        files.extend(log_dir.rglob('*error*analysis*.csv'))
        files.extend(log_dir.rglob('*oof*error*.csv'))
        files.extend(log_dir.rglob('*val*error*.csv'))
    if wandb_dir.exists():
        files.extend(wandb_dir.rglob('*error*analysis*.csv'))
        files.extend(wandb_dir.rglob('*oof*error*.csv'))
        files.extend(wandb_dir.rglob('*val*error*.csv'))

    # De-duplicate while preserving order.
    seen = set()
    uniq = []
    for f in files:
        fp = str(f.resolve())
        if fp not in seen and f.is_file():
            seen.add(fp)
            uniq.append(f)
    return uniq


def discover_wandb_summaries(wandb_dir: Path):
    if not wandb_dir.exists():
        return []
    return sorted(wandb_dir.rglob('wandb-summary.json'))


error_csv_files = discover_error_csvs(LOG_DIR, WANDB_DIR)
summary_json_files = discover_wandb_summaries(WANDB_DIR)

print(f'Found {len(error_csv_files)} candidate error CSV files')
for p in error_csv_files[:20]:
    print(' -', p)

print(f'Found {len(summary_json_files)} wandb summary files')
for p in summary_json_files[:10]:
    print(' -', p)

In [ ]:
def load_wandb_run_metrics(summary_paths):
    records = []
    for p in summary_paths:
        try:
            data = json.loads(Path(p).read_text(encoding='utf-8'))
        except Exception:
            continue

        run_name = p.parent.parent.name
        rec = {'run': run_name, 'summary_path': str(p)}

        # Keep only scalar metrics for compact reporting.
        for k, v in data.items():
            if isinstance(v, (int, float, str, bool)):
                rec[k] = v

        records.append(rec)

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    return df


def normalize_error_df(df: pd.DataFrame, source_path: Path):
    d = df.copy()

    # Harmonize required columns.
    if 'id' not in d.columns:
        if 'image_id' in d.columns:
            d['id'] = d['image_id']
        elif 'path' in d.columns:
            d['id'] = d['path'].astype(str).apply(lambda x: os.path.basename(x))

    if 'path' not in d.columns:
        d['path'] = np.nan

    if 'y_true' not in d.columns:
        d['y_true'] = np.nan
    if 'y_pred' not in d.columns:
        d['y_pred'] = np.nan

    # Probability aliases.
    if 'y_prob_meteorite' not in d.columns:
        for alt in ['prob', 'score', 'y_prob', 'pred_prob']:
            if alt in d.columns:
                d['y_prob_meteorite'] = d[alt]
                break
    if 'y_prob_meteorite' not in d.columns:
        d['y_prob_meteorite'] = np.nan

    if 'is_error' not in d.columns:
        d['is_error'] = (d['y_true'] != d['y_pred'])

    if 'error_type' not in d.columns:
        d['error_type'] = np.where(
            d['is_error'] & (d['y_true'] == 1),
            'FN_meteorite_missed',
            np.where(d['is_error'] & (d['y_true'] == 0), 'FP_false_alarm', 'correct')
        )

    if 'error_confidence' not in d.columns:
        d['error_confidence'] = np.where(
            d['y_pred'] == 1,
            d['y_prob_meteorite'],
            1.0 - d['y_prob_meteorite']
        )

    d['source_csv'] = str(source_path)
    d['source_name'] = source_path.name

    keep_cols = [
        'id', 'path', 'y_true', 'y_pred', 'y_prob_meteorite',
        'is_error', 'error_type', 'error_confidence',
        'source_csv', 'source_name'
    ]
    for c in keep_cols:
        if c not in d.columns:
            d[c] = np.nan

    d = d[keep_cols].copy()
    d['id'] = d['id'].astype(str)
    d['is_error'] = d['is_error'].fillna(False).astype(bool)
    return d


run_metrics_df = load_wandb_run_metrics(summary_json_files)
if len(run_metrics_df):
    run_metrics_df.to_csv(OUT_RUN_METRICS, index=False)
    print('Saved run metrics:', OUT_RUN_METRICS)
    display(run_metrics_df.head(10))
else:
    print('No wandb summary metrics loaded.')

In [ ]:
all_rows = []
for p in error_csv_files:
    try:
        raw = pd.read_csv(p)
    except Exception as e:
        print(f'Skip unreadable CSV: {p} | {e}')
        continue

    norm = normalize_error_df(raw, p)
    all_rows.append(norm)

if not all_rows:
    raise RuntimeError('No readable error-analysis CSV found in logs/wandb.')

errors_long = pd.concat(all_rows, axis=0, ignore_index=True)

# Optional path fill from clean index.
clean_index_path = LOG_DIR / 'clean_train_ids.csv'
if clean_index_path.exists() and errors_long['path'].isna().any():
    clean_idx = pd.read_csv(clean_index_path)
    if {'id', 'path'}.issubset(clean_idx.columns):
        id2path = dict(zip(clean_idx['id'].astype(str), clean_idx['path'].astype(str)))
        errors_long['path'] = errors_long['path'].fillna(errors_long['id'].map(id2path))

errors_only = errors_long[errors_long['is_error']].copy()
if len(errors_only) == 0:
    raise RuntimeError('No misclassified samples found in discovered CSV files.')

hard_df = errors_only.groupby('id', as_index=False).agg(
    error_count=('is_error', 'sum'),
    run_count=('source_csv', 'nunique'),
    fn_count=('error_type', lambda x: int((x == 'FN_meteorite_missed').sum())),
    fp_count=('error_type', lambda x: int((x == 'FP_false_alarm').sum())),
    mean_error_conf=('error_confidence', 'mean'),
    max_error_conf=('error_confidence', 'max'),
    mean_prob=('y_prob_meteorite', 'mean'),
    median_prob=('y_prob_meteorite', 'median'),
    path=('path', 'first'),
)

hard_df['fn_ratio'] = hard_df['fn_count'] / hard_df['error_count'].clip(lower=1)
hard_df['fp_ratio'] = hard_df['fp_count'] / hard_df['error_count'].clip(lower=1)

# Hardness score favors repeated and confident mistakes.
hard_df['hard_score'] = (
    1.2 * hard_df['error_count']
    + 0.6 * hard_df['run_count']
    + 0.8 * hard_df['mean_error_conf'].fillna(0.0)
    + 0.4 * hard_df['max_error_conf'].fillna(0.0)
)

def recommend_action(row):
    if row['error_count'] >= 4 and row['fn_ratio'] >= 0.6:
        return 'HIGH: relabel review (possible missed meteorite)'
    if row['error_count'] >= 4 and row['fp_ratio'] >= 0.6:
        return 'HIGH: hard negative mining / relabel review'
    if row['error_count'] >= 3:
        return 'MEDIUM: add to difficult-set and recheck label'
    if row['mean_error_conf'] >= 0.8:
        return 'MEDIUM: confidence mismatch, inspect annotation'
    return 'LOW: monitor only'

hard_df['recommendation'] = hard_df.apply(recommend_action, axis=1)
hard_df = hard_df.sort_values(['hard_score', 'error_count'], ascending=[False, False]).reset_index(drop=True)
hard_top = hard_df.head(TOP_K).copy()
hard_top.to_csv(OUT_CSV, index=False)

print('Total error rows loaded:', len(errors_only))
print('Unique hard samples:', len(hard_df))
print('Saved ranked hard samples:', OUT_CSV)
display(hard_top.head(20))

In [ ]:
def show_hard_examples(df, n=24, cols=6):
    view = df.head(n).copy()
    rows = int(np.ceil(len(view) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.0 * rows))

    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    elif cols == 1:
        axes = np.array([[ax] for ax in axes])

    axes_flat = axes.flatten()
    for i, ax in enumerate(axes_flat):
        if i >= len(view):
            ax.axis('off')
            continue

        row = view.iloc[i]
        p = row.get('path', None)
        title = f"#{i+1} cnt={int(row['error_count'])} score={row['hard_score']:.2f}"

        if isinstance(p, str) and p and os.path.exists(p):
            try:
                img = Image.open(p).convert('RGB')
                ax.imshow(img)
                ax.set_title(title, fontsize=9)
            except Exception:
                ax.text(0.5, 0.5, 'unreadable image', ha='center', va='center')
                ax.set_title(title, fontsize=9)
        else:
            ax.text(0.5, 0.5, str(row['id'])[:30], ha='center', va='center')
            ax.set_title(title, fontsize=9)

        ax.axis('off')

    plt.tight_layout()
    plt.show()


show_hard_examples(hard_top, n=min(DISPLAY_K, len(hard_top)), cols=6)

print('Recommendation counts:')
print(hard_top['recommendation'].value_counts())

## Suggested Use

1. Run all cells and inspect the top ranked rows in `datacleaner_hard_samples.csv`.
2. Prioritize rows tagged `HIGH` for manual relabel verification.
3. Build a cleaned split (exclude confirmed bad labels), then retrain with the CV+OOF pipeline.